# ASR Benchmark: Whisper Large V3 Turbo on VietSuperSpeech

Run this notebook on **Google Colab** with a free **T4 GPU** to benchmark the ASR model.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tranxuantruongworld/asr-benchmark/blob/init-setup/notebooks/benchmark_colab.ipynb)

**Steps:**
1. Select **Runtime > Change runtime type > T4 GPU**
2. Run all cells
3. Results will be saved and optionally pushed to HuggingFace

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets[audio] jiwer accelerate soundfile librosa huggingface_hub tqdm torch torchaudio

## 2. Check GPU Availability

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

## 3. Configuration

In [ ]:
# === CONFIGURATION ===
MODEL_ID = "openai/whisper-large-v3-turbo"
DATASET_ID = "thanhnew2001/VietSuperSpeech"
SPLIT = "validation"
LANGUAGE = "vietnamese"
BATCH_SIZE = 8  # Increase to 16 or 32 if you have more GPU memory
MAX_SAMPLES = None  # Set to a number for quick testing, None for full benchmark

# HuggingFace Hub (optional - for pushing results)
HF_REPO_ID = ""  # e.g. "your-username/asr-benchmark-results"
PUSH_TO_HF = False  # Set to True to push results to HuggingFace

## 4. Load Model

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading {MODEL_ID} on {device}...")
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)
processor = AutoProcessor.from_pretrained(MODEL_ID)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)
print("Model loaded!")

## 5. Load Dataset

In [ ]:
from datasets import load_dataset

print(f"Loading dataset {DATASET_ID} ({SPLIT})...")
dataset = load_dataset(DATASET_ID, split=SPLIT)
print(f"Dataset loaded: {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")
print(f"Sample: {dataset[0]['text'][:100]}")

## 6. Run Benchmark

In [ ]:
import io
import time
import numpy as np
import librosa
import requests
from jiwer import wer, cer
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_url

TARGET_SR = 16000

def normalize_text(text: str) -> str:
    return " ".join(text.strip().lower().split())

def download_audio(audio_path: str, dataset_id: str) -> np.ndarray:
    url = hf_hub_url(repo_id=dataset_id, filename=audio_path, repo_type="dataset")
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    audio_array, _ = librosa.load(io.BytesIO(response.content), sr=TARGET_SR, mono=True)
    return audio_array

num_samples = len(dataset) if MAX_SAMPLES is None else min(MAX_SAMPLES, len(dataset))
print(f"Running benchmark on {num_samples} samples...")

references = []
predictions = []
per_sample_results = []
total_audio_duration = 0.0
total_inference_time = 0.0

for i in tqdm(range(0, num_samples, BATCH_SIZE), desc="Benchmarking"):
    batch_end = min(i + BATCH_SIZE, num_samples)
    batch = dataset.select(list(range(i, batch_end)))

    audio_arrays = []
    ref_texts = []
    durations = []

    for sample in batch:
        try:
            audio_array = download_audio(sample["audio"], DATASET_ID)
            audio_arrays.append(audio_array)
            ref_texts.append(sample["text"])
            durations.append(sample.get("duration", 0.0))
        except Exception as e:
            print(f"Skipping sample: {e}")

    if not audio_arrays:
        continue

    start_time = time.time()
    results = pipe(
        audio_arrays,
        batch_size=len(audio_arrays),
        generate_kwargs={"language": LANGUAGE, "task": "transcribe"},
    )
    inference_time = time.time() - start_time
    total_inference_time += inference_time

    for j, (result, ref_text, dur) in enumerate(zip(results, ref_texts, durations)):
        ref_norm = normalize_text(ref_text)
        pred_norm = normalize_text(result["text"])
        sample_wer = wer(ref_norm, pred_norm) if ref_norm else 0.0
        sample_cer = cer(ref_norm, pred_norm) if ref_norm else 0.0
        total_audio_duration += dur
        references.append(ref_norm)
        predictions.append(pred_norm)
        per_sample_results.append({
            "index": i + j, "reference": ref_norm, "prediction": pred_norm,
            "wer": round(sample_wer, 4), "cer": round(sample_cer, 4),
            "duration_s": round(dur, 2),
        })

print("Benchmark complete!")

## 7. Results

In [ ]:
import json
from datetime import datetime, timezone

overall_wer = wer(references, predictions)
overall_cer = cer(references, predictions)
rtf = total_inference_time / total_audio_duration if total_audio_duration > 0 else None

metrics = {
    "overall_wer": round(overall_wer, 4),
    "overall_cer": round(overall_cer, 4),
    "num_samples": num_samples,
    "total_audio_duration_s": round(total_audio_duration, 2),
    "total_inference_time_s": round(total_inference_time, 2),
    "real_time_factor": round(rtf, 4) if rtf else None,
    "batch_size": BATCH_SIZE,
}

print("=" * 50)
print("BENCHMARK RESULTS")
print("=" * 50)
print(f"WER:  {metrics['overall_wer']:.4f} ({metrics['overall_wer']*100:.2f}%)")
print(f"CER:  {metrics['overall_cer']:.4f} ({metrics['overall_cer']*100:.2f}%)")
print(f"Samples:       {metrics['num_samples']}")
print(f"Audio:         {metrics['total_audio_duration_s']:.1f}s")
print(f"Inference:     {metrics['total_inference_time_s']:.1f}s")
if rtf:
    print(f"RTF:           {metrics['real_time_factor']:.4f}")
print("=" * 50)

# Save full results
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
result_payload = {
    "metadata": {
        "model_id": MODEL_ID, "dataset_id": DATASET_ID, "split": SPLIT,
        "language": LANGUAGE, "timestamp": timestamp,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "torch_version": torch.__version__, "max_samples": MAX_SAMPLES,
    },
    "metrics": metrics,
    "per_sample_results": per_sample_results,
}

output_file = f"benchmark_results_{timestamp}.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result_payload, f, ensure_ascii=False, indent=2)
print(f"\nResults saved to {output_file}")

## 8. Error Analysis

In [ ]:
import pandas as pd

df = pd.DataFrame(per_sample_results)

print("\n--- WER/CER Statistics ---")
print(df[["wer", "cer", "duration_s"]].describe())

print("\n--- Top 5 WORST Samples (highest WER) ---")
worst = df.nlargest(5, "wer")
for _, row in worst.iterrows():
    print(f"\n[Sample {row['index']}] WER={row['wer']:.4f} CER={row['cer']:.4f}")
    print(f"  REF: {row['reference'][:100]}")
    print(f"  PRD: {row['prediction'][:100]}")

print("\n--- Top 5 BEST Samples (lowest WER) ---")
best = df.nsmallest(5, "wer")
for _, row in best.iterrows():
    print(f"\n[Sample {row['index']}] WER={row['wer']:.4f} CER={row['cer']:.4f}")
    print(f"  REF: {row['reference'][:100]}")
    print(f"  PRD: {row['prediction'][:100]}")

## 9. Push Results to HuggingFace (Optional)

In [ ]:
if PUSH_TO_HF and HF_REPO_ID:
    from huggingface_hub import notebook_login, HfApi, create_repo
    from datasets import Dataset, DatasetDict, Features, Value

    # Login to HuggingFace
    notebook_login()

    # Create repo
    create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)

    # Push summary
    summary_ds = Dataset.from_dict({
        "model_id": [MODEL_ID], "dataset_id": [DATASET_ID],
        "split": [SPLIT], "language": [LANGUAGE], "timestamp": [timestamp],
        "device": ["cuda" if torch.cuda.is_available() else "cpu"],
        "wer": [metrics["overall_wer"]], "cer": [metrics["overall_cer"]],
        "num_samples": [metrics["num_samples"]],
        "total_audio_duration_s": [metrics["total_audio_duration_s"]],
        "total_inference_time_s": [metrics["total_inference_time_s"]],
        "real_time_factor": [metrics.get("real_time_factor")],
        "batch_size": [metrics["batch_size"]],
    })

    # Push per-sample
    samples_ds = Dataset.from_dict({
        "index": [s["index"] for s in per_sample_results],
        "reference": [s["reference"] for s in per_sample_results],
        "prediction": [s["prediction"] for s in per_sample_results],
        "wer": [s["wer"] for s in per_sample_results],
        "cer": [s["cer"] for s in per_sample_results],
        "duration_s": [s["duration_s"] for s in per_sample_results],
    })

    ds_dict = DatasetDict({"summary": summary_ds, "per_sample": samples_ds})
    ds_dict.push_to_hub(HF_REPO_ID)
    print(f"Results pushed to https://huggingface.co/datasets/{HF_REPO_ID}")
else:
    print("Skipping push to HuggingFace. Set PUSH_TO_HF=True and HF_REPO_ID to enable.")
    print(f"Download results: {output_file}")

## 10. Download Results

In [ ]:
from google.colab import files
files.download(output_file)
print(f"Downloaded {output_file}")